# RAG Evaluation Report — Apple 2025 10-K

This notebook visualises the pre-computed evaluation results from `data/eval/` across two layers:

## Part 1 — Retrieval Evaluation (all 4 strategies, 50 gold questions)

| Strategy | Description |
|----------|-------------|
| `semantic` | Dense OpenAI `text-embedding-3-small` embeddings + Chroma cosine search |
| `bm25` | BM25Okapi lexical retrieval |
| `hybrid` | Reciprocal Rank Fusion (k=60) of semantic + BM25 |
| `hybrid_rerank` | Hybrid RRF (top-20 pool) reranked by `cross-encoder/ms-marco-MiniLM-L-6-v2` |

> **Run `scripts/run_retrieval_eval.py` first** to regenerate `retrieval_results.json` / `retrieval_summary.json` if missing.

## Part 2 — Generation Quality (RAGAS, 20 questions, GPT-4o-mini)

| Metric | What it measures |
|--------|-----------------|
| Faithfulness | Claims in the answer supported by retrieved context |
| Answer Relevancy | Answer is on-topic and addresses the question |
| Context Recall | All necessary information was retrieved |
| Context Precision | Retrieved context is relevant (low noise) |

> **Run `scripts/run_ragas_eval.py` first** to regenerate `ragas_results.json` / `ragas_summary.json` if missing.

In [ ]:
import json
from pathlib import Path

ROOT = Path().resolve().parent
SUMMARY_PATH = ROOT / 'data' / 'eval' / 'retrieval_summary.json'
RESULTS_PATH = ROOT / 'data' / 'eval' / 'retrieval_results.json'

with open(SUMMARY_PATH) as f:
    summaries = json.load(f)

with open(RESULTS_PATH) as f:
    results = json.load(f)

print(f'Loaded {len(summaries)} strategies, {summaries[0]["n_questions"]} questions each')

## Summary metrics table

In [ ]:
import pandas as pd

df = pd.DataFrame(summaries).set_index('strategy')
df = df[['recall@1','recall@5','recall@10','mrr',
         'primary_recall@1','primary_recall@5','primary_recall@10','primary_mrr',
         'mean_latency_ms']]
df.columns = ['R@1','R@5','R@10','MRR','pR@1','pR@5','pR@10','pMRR','ms/q']
df.style.highlight_max(axis=0, color='#c6efce').format('{:.4f}', subset=['R@1','R@5','R@10','MRR','pR@1','pR@5','pR@10','pMRR']).format('{:.1f}', subset='ms/q')

## Recall@k bar chart

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

strategies = [s['strategy'] for s in summaries]
r1  = [s['recall@1']  for s in summaries]
r5  = [s['recall@5']  for s in summaries]
r10 = [s['recall@10'] for s in summaries]

x = np.arange(len(strategies))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width, r1,  width, label='R@1',  color='#4472C4')
bars2 = ax.bar(x,         r5,  width, label='R@5',  color='#ED7D31')
bars3 = ax.bar(x + width, r10, width, label='R@10', color='#A9D18E')

for bars in (bars1, bars2, bars3):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(strategies, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Recall')
ax.set_title('Recall@k by Retrieval Strategy (Apple 2025 10-K, 50 gold questions)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks' / 'recall_at_k.png', dpi=150)
plt.show()

## MRR comparison

In [ ]:
mrr_any  = [s['mrr']         for s in summaries]
mrr_prim = [s['primary_mrr'] for s in summaries]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width/2, mrr_any,  width, label='MRR (any gold)',     color='#5B9BD5')
ax.bar(x + width/2, mrr_prim, width, label='MRR (primary gold)', color='#FF7F50')

for i, (a, p) in enumerate(zip(mrr_any, mrr_prim)):
    ax.text(i - width/2, a + 0.01, f'{a:.3f}', ha='center', va='bottom', fontsize=8)
    ax.text(i + width/2, p + 0.01, f'{p:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(strategies, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_ylabel('MRR')
ax.set_title('Mean Reciprocal Rank by Strategy')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks' / 'mrr_comparison.png', dpi=150)
plt.show()

## Latency vs quality trade-off

In [ ]:
latencies = [s['mean_latency_ms'] for s in summaries]
colors    = ['#4472C4', '#ED7D31', '#A9D18E', '#FF6B6B']

fig, ax = plt.subplots(figsize=(8, 5))
for i, s in enumerate(summaries):
    ax.scatter(s['mean_latency_ms'], s['recall@5'], s=180, color=colors[i],
               zorder=5, label=s['strategy'])
    ax.annotate(s['strategy'],
                (s['mean_latency_ms'], s['recall@5']),
                textcoords='offset points', xytext=(8, 4), fontsize=9)

ax.set_xlabel('Mean latency (ms/query)')
ax.set_ylabel('Recall@5')
ax.set_title('Quality vs Latency Trade-off')
ax.set_xlim(-50, max(latencies) * 1.15)
ax.set_ylim(0.5, 1.0)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks' / 'latency_vs_quality.png', dpi=150)
plt.show()

## Per-question hit analysis

In [ ]:
# For each question, show which strategies found a gold chunk in top-5

def hit_in_top5(hit_ids, gold_ids):
    return any(h in set(gold_ids) for h in hit_ids[:5])

strategy_names = list(results.keys())
rows = []
for i, qrow in enumerate(results[strategy_names[0]]['per_question']):
    row = {'question_id': qrow['question_id'], 'question': qrow['question'][:70] + '...'}
    for sname in strategy_names:
        pq = results[sname]['per_question'][i]
        row[sname] = 'Y' if hit_in_top5(pq['hit_ids'], pq['gold_ids']) else ''
    rows.append(row)

df_hits = pd.DataFrame(rows).set_index('question_id')

def colour_hit(val):
    return 'background-color: #c6efce' if val == 'Y' else 'background-color: #ffc7ce'

df_hits.style.applymap(colour_hit, subset=strategy_names)

## Failure analysis — questions where every strategy missed top-5

In [ ]:
misses = [r for r in rows if all(r[s] == '' for s in strategy_names)]
print(f'{len(misses)} questions where ALL strategies missed R@5:\n')
for m in misses:
    print(f"  [{m['question_id']}] {m['question']}")

## Wins unique to hybrid_rerank

In [ ]:
rerank_only = [
    r for r in rows
    if r.get('hybrid_rerank') == 'Y'
    and all(r.get(s, '') != 'Y' for s in strategy_names if s != 'hybrid_rerank')
]
print(f'{len(rerank_only)} questions where only hybrid_rerank found the answer in top-5:\n')
for r in rerank_only:
    print(f"  [{r['question_id']}] {r['question']}")

---

## Part 2 — RAGAS Generation Evaluation

Scores below are for 20 questions from the Apple 2025 gold set,  
retrieved with `hybrid_rerank` and answered by `gpt-4o-mini`.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path().resolve().parent
RAGAS_SUMMARY = ROOT / 'data' / 'eval' / 'ragas_summary.json'
RAGAS_RESULTS = ROOT / 'data' / 'eval' / 'ragas_results.json'

with open(RAGAS_SUMMARY) as f:
    ragas_summary = json.load(f)

with open(RAGAS_RESULTS) as f:
    ragas_records = json.load(f)

print(f"RAGAS summary ({len(ragas_records)} questions evaluated):")
for k, v in ragas_summary.items():
    print(f"  {k:25s}  {v:.4f}")

## RAGAS summary bar chart

In [ ]:
metric_labels = {
    'faithfulness':      'Faithfulness',
    'answer_relevancy':  'Answer\nRelevancy',
    'context_recall':    'Context\nRecall',
    'context_precision': 'Context\nPrecision',
}
metrics = list(ragas_summary.keys())
scores  = [ragas_summary[m] for m in metrics]
labels  = [metric_labels.get(m, m) for m in metrics]
colors  = ['#4472C4', '#ED7D31', '#A9D18E', '#FF6B6B']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, scores, color=colors, width=0.5, edgecolor='white')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylim(0, 1.1)
ax.set_ylabel('Score (0–1)', fontsize=11)
ax.set_title('RAGAS Generation Evaluation\nhybrid_rerank + GPT-4o-mini | Apple 2025 10-K | 20 questions',
             fontsize=11)
ax.axhline(0.8, color='gray', linestyle='--', linewidth=0.8, alpha=0.6, label='0.8 threshold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks' / 'ragas_scores.png', dpi=150)
plt.show()

## Per-question RAGAS scores heatmap

In [ ]:
ragas_rows = []
for rec in ragas_records:
    r = rec.get('ragas', {})
    ragas_rows.append({
        'Q':                   rec['question_id'],
        'Faithfulness':        r.get('faithfulness'),
        'Answer Relevancy':    r.get('answer_relevancy'),
        'Context Recall':      r.get('context_recall'),
        'Context Precision':   r.get('context_precision'),
    })

df_ragas = pd.DataFrame(ragas_rows).set_index('Q')
df_ragas.style \
    .background_gradient(cmap='RdYlGn', vmin=0, vmax=1) \
    .format('{:.3f}', na_rep='--')

In [ ]:
## Low-faithfulness questions (faithfulness < 0.8)
questions_low_faith = [
    r for r in ragas_records
    if r.get('ragas', {}).get('faithfulness') is not None
    and r['ragas']['faithfulness'] < 0.8
]
print(f"{len(questions_low_faith)} questions with faithfulness < 0.8:\n")
for r in questions_low_faith:
    print(f"  [{r['question_id']}] faith={r['ragas']['faithfulness']:.3f}  {r['question'][:80]}")
    print(f"    Answer snippet: {r['answer'][:120]}...")
    print()